# M31 lab solution: Two-tool agent

**Brief.** Give an agent a private-docs RAG tool and a web-search tool, and tune the tool descriptions until it picks the right one on 20 mixed questions.

`../data/mixed_questions.jsonl` labels each of 20 questions with the tool it needs, `docs` or `web`. Each round changes only the two tool descriptions; the system prompt, the model and the questions stay fixed. Tool choice is the first tool call the model makes, which is the agent's first decision in the loop of lesson 31.1.

The notebook reads only `../data/` and imports only from `src/`. Run it from this folder; the lab README lists the keys it needs.

## Setup

In [ ]:
import json
import sys
from pathlib import Path

import pandas as pd
from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(usecwd=True))
sys.path.insert(0, str(Path("../../../../src").resolve()))

from helpers import get_embeddings, get_experientiallabs_llm
from lab_kit import is_hit, load_handbook, load_jsonl

DATA = Path("../data")
llm = get_experientiallabs_llm()
print("LLM:", llm.model_name)

## The Part V system, rebuilt inline

This lab does not depend on another lab's output, so the cell below rebuilds the minimal Part V system: the handbook loaded with `lab_kit`, the baseline chunker (500 characters, 50 overlap), an in-memory vector store over `get_embeddings()`, and an answer prompt that cites passages by number. Later cells reuse `retriever`, `format_docs` and `rag_answer`.

In [ ]:
from langchain_core.documents import Document
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter

handbook = load_handbook(DATA / "larkspur" / "handbook")
golden = load_jsonl(DATA / "larkspur" / "golden.jsonl")
splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
chunks = [Document(page_content=c, metadata={"source": p["source"]}) for p in handbook for c in splitter.split_text(p["text"])]
embeddings = get_embeddings()
store = InMemoryVectorStore.from_documents(chunks, embeddings)
retriever = store.as_retriever(search_kwargs={"k": 4})

ANSWER_PROMPT = ChatPromptTemplate.from_messages([
    ("system", "You answer questions about the Larkspur Cloud handbook. Use only the numbered passages. "
               "Cite each claim with its passage number in square brackets, like [2]. "
               "If the passages do not contain the answer, reply: The handbook does not say."),
    ("human", "Passages:\n{context}\n\nQuestion: {question}"),
])
answer_chain = ANSWER_PROMPT | llm | StrOutputParser()


def format_docs(docs):
    return "\n\n".join(f"[{i}] ({d.metadata['source']}) {d.page_content}" for i, d in enumerate(docs, 1))


def rag_answer(question, docs=None):
    """Retrieve (unless docs are given) and answer with citations; returns (answer, docs)."""
    docs = retriever.invoke(question) if docs is None else docs
    return answer_chain.invoke({"context": format_docs(docs), "question": question}), docs


print(len(chunks), "chunks;", len(golden), "golden questions")

In [ ]:
questions = load_jsonl(DATA / "mixed_questions.jsonl")
print(pd.Series([q["tool"] for q in questions]).value_counts().to_string())

## 1. The two tools' functions

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage
from langchain_core.tools import StructuredTool
from langchain_tavily import TavilySearch

tavily = TavilySearch(max_results=3)


def search_docs(query: str) -> str:
    """Search the private documents."""
    return format_docs(retriever.invoke(query))


def search_web(query: str) -> str:
    """Search the web."""
    results = tavily.invoke({"query": query}).get("results", [])
    return "\n\n".join(f"{r['url']}\n{r['content']}" for r in results)


SYSTEM = "You help Larkspur Cloud's support engineers. Look things up with a tool before you answer."


def make_tools(docs_description, web_description):
    return [StructuredTool.from_function(search_docs, name="search_docs", description=docs_description),
            StructuredTool.from_function(search_web, name="search_web", description=web_description)]

## 2. Three rounds of tool descriptions

In [ ]:
ROUNDS = [
    {"round": 1,
     "docs": "Search documents.",
     "web": "Search the internet."},
    {"round": 2,
     "docs": ("Search the Larkspur Cloud customer handbook: billing, refunds, plans and server sizes, outages, backups, "
              "data export, account deletion, security, API limits, support tiers, regions, SLA credits and onboarding."),
     "web": "Search the public web for general technical knowledge, standards and current events."},
    {"round": 3,
     "docs": ("Search the Larkspur Cloud customer handbook: billing, refunds, plans and server sizes, outages, backups, "
              "data export, account deletion, security, API limits, support tiers, regions, SLA credits and onboarding. "
              "Use it for every question about Larkspur's own policies, limits or procedures, even when the question uses "
              "a general term such as HTTP 429, AES-256 or EU data residency."),
     "web": ("Search the public web. Use it for standards (RFCs, laws), software releases, other providers and general "
             "engineering practice. Never use it for Larkspur's own policies: the web has no Larkspur documentation.")},
]


def first_tool(model, question):
    msg = model.invoke([SystemMessage(SYSTEM), HumanMessage(question)])
    return msg.tool_calls[0]["name"] if msg.tool_calls else "none"


rows = []
for r in ROUNDS:
    chooser = llm.bind_tools(make_tools(r["docs"], r["web"]))
    print(f"Round {r['round']}\n  search_docs: {r['docs']}\n  search_web:  {r['web']}")
    for q in questions:
        picked = first_tool(chooser, q["question"])
        rows.append({"round": r["round"], "id": q["id"], "label": q["tool"], "picked": picked,
                     "correct": picked == f"search_{q['tool']}"})
choices = pd.DataFrame(rows)
print(choices.pivot(index=["id", "label"], columns="round", values="picked").to_string())

## 3. Results

In [ ]:
summary = choices.groupby("round").agg(
    accuracy=("correct", "mean"),
    docs_correct=("correct", lambda s: s[choices.loc[s.index, "label"] == "docs"].sum()),
    web_correct=("correct", lambda s: s[choices.loc[s.index, "label"] == "web"].sum()),
    no_tool=("picked", lambda s: (s == "none").sum()),
)
print(summary.to_string())
last = choices[choices["round"] == ROUNDS[-1]["round"]]
print("\nStill wrong in the last round:", ", ".join(last[~last.correct].id) or "none")

## 4. The full agent with the last round's descriptions

The same tools inside `create_agent`, with a model-call cap, on one question of each kind.

In [ ]:
from langchain.agents import create_agent
from langchain.agents.middleware import ModelCallLimitMiddleware

agent = create_agent(llm, make_tools(ROUNDS[-1]["docs"], ROUNDS[-1]["web"]), system_prompt=SYSTEM,
                     middleware=[ModelCallLimitMiddleware(run_limit=4)])
for q in (next(q for q in questions if q["tool"] == "docs"), next(q for q in questions if q["tool"] == "web")):
    result = agent.invoke({"messages": [{"role": "user", "content": q["question"]}]})
    calls = [tc["name"] for m in result["messages"] for tc in (getattr(m, "tool_calls", None) or [])]
    print(f"{q['question']}\n  tools called: {calls}\n  answer: {result['messages'][-1].content[:300]}\n")

## What this shows

- The summary is accuracy per round of descriptions, with the descriptions printed above it, which is what the brief asks for.
- Only the descriptions changed between rounds, so any change in accuracy belongs to them. Read the per-question table for the questions that flip: they show which words in a description did the work.
- A question that names a general term (HTTP 429, AES-256) can belong to either tool. Saying so in the description is cheaper than a router, and the questions still wrong in the last round are where a router or a clarifying question would start to pay.